# NER Notebook 2: Training & Evaluation

**BSAN 6200 — Text Mining & Social Media Analytics**

## What You'll Learn
1. **CRF training** — the classic ML approach to sequence labeling
2. **Bi-LSTM training** — neural network approach
3. **Transformer fine-tuning** — modern state-of-the-art with DistilBERT
4. **Entity-level evaluation** with seqeval (the right way to measure NER)
5. **Error analysis** — categorize and diagnose model failures

## Prerequisites
- Notebook 1 (NER inference and data formats)
- Basic understanding of train/test splits and classification metrics

---

In [ ]:
# ============================================================
# SETUP: Install all required packages
# ============================================================
# sklearn-crfsuite: CRF implementation for sequence labeling
# seqeval: entity-level evaluation metrics (the correct way to evaluate NER)
# transformers: HuggingFace library for Transformer models
# datasets: HuggingFace datasets library for loading NER benchmarks

#!pip install sklearn-crfsuite seqeval transformers datasets accelerate tensorflow

In [2]:
import sklearn_crfsuite
from sklearn_crfsuite import metrics as crf_metrics
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report as seq_report
from seqeval.metrics import f1_score as seq_f1
from seqeval.scheme import IOB2
from sklearn.metrics import classification_report as sklearn_report
import pandas as pd
import numpy as np
import json

print("All packages imported!")

All packages imported!


---
## Part 1: Conditional Random Fields (CRF)

CRF is a probabilistic model for **sequence labeling**. Think of it as a
"smart proofreader" that looks at neighboring tokens to ensure label
sequences are consistent.

**How CRF works for NER:**
1. Extract features for each token (is it capitalized? what's the previous word?)
2. Learn transition patterns from training data (e.g., B-PER is often followed by I-PER)
3. At prediction time, find the most likely label sequence considering both features AND transitions

**Pros:** Fast, interpretable, works with small data
**Cons:** Requires hand-crafted features, can't handle unseen words well

In [4]:
# ============================================================
# LOAD TRAINING DATA: CoNLL format
# ============================================================
# We'll use the WNUT17 dataset (Emerging entities from social media)
# This dataset has entities that are harder to detect: new companies,
# creative works, products, etc.

def read_conll(filename):
    """
    Read a CoNLL-formatted file into a list of sentences.

    Each sentence is a list of (token, label) tuples.
    Sentences are separated by blank lines in the file.

    Returns:
        List[List[Tuple[str, str]]]: list of sentences,
        where each sentence is a list of (token, tag) pairs
    """
    sentences = []
    with open(filename, encoding="utf-8") as f:
        sentence = []
        for line in f:
            line = line.strip()
            if line:
                # Each non-empty line has: token<tab>label
                parts = line.split("\t")
                if len(parts) >= 2:
                    sentence.append((parts[0], parts[1]))
            else:
                # Blank line = end of sentence
                if sentence:
                    sentences.append(sentence)
                    sentence = []
        # Don't forget the last sentence (if file doesn't end with blank line)
        if sentence:
            sentences.append(sentence)
    return sentences

# Download WNUT17 data
import urllib.request
import os

wnut_url = "https://raw.githubusercontent.com/leondz/emerging_entities_17/master/wnut17train.conll"
if not os.path.exists("wnut17train.conll"):
    urllib.request.urlretrieve(wnut_url, "wnut17train.conll")
    print("Downloaded WNUT17 training data")

sentences = read_conll("wnut17train.conll")
print(f"Loaded {len(sentences)} sentences")
print(f"\nExample sentence:")
for token, tag in sentences[0]:
    marker = "  ←" if tag != "O" else ""
    print(f"  {token:<20} {tag}{marker}")

Downloaded WNUT17 training data
Loaded 3394 sentences

Example sentence:
  @paulwalk            O
  It                   O
  's                   O
  the                  O
  view                 O
  from                 O
  where                O
  I                    O
  'm                   O
  living               O
  for                  O
  two                  O
  weeks                O
  .                    O
  Empire               B-location  ←
  State                I-location  ←
  Building             I-location  ←
  =                    O
  ESB                  B-location  ←
  .                    O
  Pretty               O
  bad                  O
  storm                O
  here                 O
  last                 O
  evening              O
  .                    O


In [5]:
# ============================================================
# FEATURE ENGINEERING: Define what the CRF sees for each token
# ============================================================
# CRF can't read raw text — it needs numeric/boolean features.
# We manually define features that help distinguish entities:
#   - Word shape (capitalized? all caps? has digits?)
#   - Suffixes (last 2-3 characters help identify word type)
#   - Context (what are the neighboring words?)

def word2features(sent, i):
    """
    Extract features for the i-th token in a sentence.

    Features capture:
    1. Properties of the current word (shape, suffix, case)
    2. Properties of the previous word (context from the left)
    3. Properties of the next word (context from the right)
    4. Position markers (beginning/end of sentence)

    Args:
        sent: list of (token, tag) tuples
        i: index of the current token

    Returns:
        dict: feature name → feature value
    """
    word = sent[i][0]

    features = {
        # === Current word features ===
        'bias': 1.0,                    # constant feature (baseline)
        'word.lower()': word.lower(),   # lowercased word (captures identity)
        'word[-3:]': word[-3:],         # last 3 chars (suffix: "-tion", "-ing")
        'word[-2:]': word[-2:],         # last 2 chars
        'word.isupper()': word.isupper(),    # ALL CAPS? (acronyms like NASA, FBI)
        'word.istitle()': word.istitle(),    # Title Case? (names: "Thomas", "Google")
        'word.isdigit()': word.isdigit(),    # Pure number? ("2024", "500")
        'word.isalpha()': word.isalpha(),    # All letters? (vs. "3rd", "U.S.")
        'word.length': len(word),            # Word length
        'word.has_hyphen': '-' in word,      # Hyphenated? ("Manuel-Miranda")
        'word.has_dollar': '$' in word,      # Dollar sign? ("$500")
        'word.has_at': '@' in word,          # At sign? (Twitter handles)
    }

    # === Previous word features (left context) ===
    if i > 0:
        word1 = sent[i-1][0]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
        })
    else:
        features['BOS'] = True  # Beginning Of Sentence marker

    # === Next word features (right context) ===
    if i < len(sent) - 1:
        word1 = sent[i+1][0]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
        })
    else:
        features['EOS'] = True  # End Of Sentence marker

    return features

# Helper functions to convert full sentences
def sent2features(sent):
    """Extract features for all tokens in a sentence."""
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    """Extract labels for all tokens in a sentence."""
    return [label for token, label in sent]

def sent2tokens(sent):
    """Extract tokens for all tokens in a sentence."""
    return [token for token, label in sent]

# Show example features
print("Features for 'Thomas' (first token):")
features = word2features(sentences[0], 0)
for name, value in list(features.items())[:10]:
    print(f"  {name:<25} = {value}")
print(f"  ... ({len(features)} features total)")

Features for 'Thomas' (first token):
  bias                      = 1.0
  word.lower()              = @paulwalk
  word[-3:]                 = alk
  word[-2:]                 = lk
  word.isupper()            = False
  word.istitle()            = False
  word.isdigit()            = False
  word.isalpha()            = False
  word.length               = 9
  word.has_hyphen           = False
  ... (16 features total)


In [6]:
# ============================================================
# TRAIN/TEST SPLIT AND CRF TRAINING
# ============================================================

# Split data: 80% train, 20% test
train_sents, test_sents = train_test_split(
    sentences, test_size=0.2, random_state=42
)
print(f"Train: {len(train_sents)} sentences")
print(f"Test:  {len(test_sents)} sentences")

# Extract features and labels
X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]
X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

# Train the CRF model
# c1 and c2 are regularization parameters (prevent overfitting)
# algorithm='lbfgs' is a standard optimization method
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,               # L1 regularization
    c2=0.1,               # L2 regularization
    max_iterations=100,    # training iterations
    all_possible_transitions=True,  # consider all label transitions
)

print("Training CRF model...")
crf.fit(X_train, y_train)
print("Training complete!")

Train: 2715 sentences
Test:  679 sentences
Training CRF model...
Training complete!


---
## Part 2: NER Evaluation — The Right Way

### Why Token-Level Accuracy is WRONG for NER

Most tokens are "O" (not entities). A model that predicts "O" for everything
would get ~90% accuracy! This is why we need **entity-level** evaluation.

### Entity-Level Evaluation with seqeval

seqeval evaluates whether the **complete entity span** was correctly predicted:
- "Thomas Jefferson" as PERSON → correct only if BOTH tokens are labeled correctly
- "Thomas" alone → WRONG (partial match = miss in strict mode)

**Always use seqeval for NER evaluation. Never use sklearn accuracy.**

In [7]:
# ============================================================
# WRONG WAY: Token-level evaluation (DO NOT USE for NER!)
# ============================================================
# This inflates scores because most tokens are "O"

y_pred = crf.predict(X_test)

# Flatten to individual tokens for sklearn
y_true_flat = [tag for sent in y_test for tag in sent]
y_pred_flat = [tag for sent in y_pred for tag in sent]

print("=" * 60)
print("⚠️  TOKEN-LEVEL METRICS (sklearn) — MISLEADING!")
print("=" * 60)

# Count how many tokens are "O"
o_count = y_true_flat.count("O")
total = len(y_true_flat)
print(f"\n{o_count}/{total} tokens ({o_count/total*100:.1f}%) are 'O' — this inflates accuracy!")

# Show the misleading accuracy
accuracy = sum(t == p for t, p in zip(y_true_flat, y_pred_flat)) / len(y_true_flat)
print(f"Token accuracy: {accuracy:.4f} — looks great, but is it?\n")

⚠️  TOKEN-LEVEL METRICS (sklearn) — MISLEADING!

12128/12725 tokens (95.3%) are 'O' — this inflates accuracy!
Token accuracy: 0.9683 — looks great, but is it?



In [8]:
# ============================================================
# RIGHT WAY: Entity-level evaluation with seqeval
# ============================================================
# seqeval evaluates COMPLETE entity spans, not individual tokens

print("=" * 60)
print("✓  ENTITY-LEVEL METRICS (seqeval) — CORRECT for NER")
print("=" * 60)
print(seq_report(y_test, y_pred))

# Also get the overall F1 score
overall_f1 = seq_f1(y_test, y_pred)
print(f"Overall entity-level F1: {overall_f1:.4f}")
print(f"\nNotice: Entity F1 is much lower than token accuracy!")
print("This is the TRUE measure of NER performance.")

✓  ENTITY-LEVEL METRICS (seqeval) — CORRECT for NER
               precision    recall  f1-score   support

  corporation       0.95      0.51      0.67        39
creative-work       0.73      0.27      0.39        30
        group       0.79      0.18      0.29        62
     location       0.79      0.51      0.62       117
       person       0.79      0.36      0.49       136
      product       0.00      0.00      0.00        20

    micro avg       0.80      0.37      0.50       404
    macro avg       0.67      0.31      0.41       404
 weighted avg       0.76      0.37      0.48       404

Overall entity-level F1: 0.5025

Notice: Entity F1 is much lower than token accuracy!
This is the TRUE measure of NER performance.


In [9]:
# ============================================================
# ERROR ANALYSIS: What types of mistakes is the model making?
# ============================================================
# Understanding errors helps you improve the model:
# - Missed entities → need more training examples
# - Spurious entities → need negative examples
# - Type mismatches → need clearer annotation guidelines

def extract_entities(tags, tokens):
    """Extract entities as (text, label) tuples from IOB tags."""
    entities = []
    current_tokens, current_label = [], None

    for token, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            if current_label:
                entities.append((" ".join(current_tokens), current_label))
            current_tokens = [token]
            current_label = tag[2:]
        elif tag.startswith("I-") and current_label:
            current_tokens.append(token)
        else:
            if current_label:
                entities.append((" ".join(current_tokens), current_label))
            current_tokens, current_label = [], None

    if current_label:
        entities.append((" ".join(current_tokens), current_label))
    return set(entities)

# Categorize errors
missed, spurious, type_mismatch = [], [], []

for sent, true_tags, pred_tags in zip(test_sents, y_test, y_pred):
    tokens = sent2tokens(sent)
    true_ents = extract_entities(true_tags, tokens)
    pred_ents = extract_entities(pred_tags, tokens)

    true_texts = {e[0] for e in true_ents}
    pred_texts = {e[0] for e in pred_ents}

    for ent in true_ents - pred_ents:
        if ent[0] in pred_texts:
            pred_label = [p[1] for p in pred_ents if p[0] == ent[0]][0]
            type_mismatch.append({"entity": ent[0], "true": ent[1], "pred": pred_label})
        else:
            missed.append({"entity": ent[0], "label": ent[1]})

    for ent in pred_ents - true_ents:
        if ent[0] not in true_texts:
            spurious.append({"entity": ent[0], "label": ent[1]})

print("ERROR ANALYSIS")
print("=" * 60)
print(f"\nMISSED entities (False Negatives): {len(missed)}")
for ex in missed[:5]:
    print(f"  • \"{ex['entity']}\" ({ex['label']})")

print(f"\nSPURIOUS entities (False Positives): {len(spurious)}")
for ex in spurious[:5]:
    print(f"  • \"{ex['entity']}\" ({ex['label']})")

print(f"\nTYPE MISMATCHES (wrong label): {len(type_mismatch)}")
for ex in type_mismatch[:5]:
    print(f"  • \"{ex['entity']}\" — True: {ex['true']}, Predicted: {ex['pred']}")

ERROR ANALYSIS

MISSED entities (False Negatives): 233
  • "auburn" (location)
  • "boilers" (group)
  • "Wonder Woman" (person)
  • "sanchez" (person)
  • "betos" (person)

SPURIOUS entities (False Positives): 21
  • "Prison" (location)
  • "Allah" (person)
  • "Rocks" (group)
  • "Launches Vivaldi" (person)
  • "T" (location)

TYPE MISMATCHES (wrong label): 15
  • "Marvel" — True: corporation, Predicted: person
  • "Ashley" — True: person, Predicted: location
  • "London Irish" — True: group, Predicted: location
  • "Baka Boyz" — True: group, Predicted: person
  • "Marie Clair" — True: corporation, Predicted: person


---
## Part 3: Bi-LSTM for NER

Bi-LSTM (Bidirectional Long Short-Term Memory) is a neural network that
reads text in **both directions** to capture full context.

Think of it as a **two-way translator**: one reads left-to-right, another reads
right-to-left, and they combine their understanding for each token.

**Key advantage over CRF:** Bi-LSTM learns its own features from the data
(no manual feature engineering needed).

In [10]:
# ============================================================
# Bi-LSTM TRAINING
# ============================================================
# We build: Input → Embedding → Bi-LSTM → Dense → Softmax → Tags

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Embedding, TimeDistributed, Dense, Bidirectional
)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# --- Step 1: Build vocabulary and tag mappings ---
# We need to convert words and tags to integer IDs

# Collect all unique words and tags
all_words = set()
all_tags = set()
for sent in sentences:
    for word, tag in sent:
        all_words.add(word)
        all_tags.add(tag)

# Create word → ID and tag → ID mappings
# Reserve ID 0 for padding, so actual words start at 1
word2idx = {w: i + 1 for i, w in enumerate(sorted(all_words))}
tag2idx = {t: i for i, t in enumerate(sorted(all_tags))}
idx2tag = {i: t for t, i in tag2idx.items()}

n_words = len(word2idx) + 1  # +1 for padding
n_tags = len(tag2idx)

print(f"Vocabulary size: {n_words}")
print(f"Number of tags: {n_tags}")
print(f"Tags: {list(tag2idx.keys())}")

# --- Step 2: Convert sentences to padded sequences ---
# Neural networks need fixed-length inputs, so we pad shorter sentences
MAX_LEN = 75  # maximum sentence length

X = [[word2idx.get(w, 0) for w, t in s] for s in sentences]
y = [[tag2idx[t] for w, t in s] for s in sentences]

# Pad sequences to MAX_LEN (shorter sentences get zeros appended)
X = pad_sequences(X, maxlen=MAX_LEN, padding='post', value=0)
y = pad_sequences(y, maxlen=MAX_LEN, padding='post', value=tag2idx.get("O", 0))

# One-hot encode the tags (required for categorical crossentropy loss)
y = np.array([to_categorical(seq, num_classes=n_tags) for seq in y])

# Train/test split
X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining samples: {len(X_train_nn)}")
print(f"Test samples: {len(X_test_nn)}")
print(f"Sequence length: {MAX_LEN}")

c:\Users\avo9\AppData\Local\anaconda3\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\avo9\AppData\Local\anaconda3\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\avo9\AppData\Local\anaconda3\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode

Vocabulary size: 14879
Number of tags: 13
Tags: ['B-corporation', 'B-creative-work', 'B-group', 'B-location', 'B-person', 'B-product', 'I-corporation', 'I-creative-work', 'I-group', 'I-location', 'I-person', 'I-product', 'O']

Training samples: 2715
Test samples: 679
Sequence length: 75


In [11]:
# --- Step 3: Build the Bi-LSTM model ---
# Architecture:
#   Input (word IDs) → Embedding (dense vectors) → Bi-LSTM (context) → Dense (per-token classification)

EMBEDDING_DIM = 50  # size of word embeddings
LSTM_UNITS = 64     # number of LSTM hidden units

# Input layer: receives sequences of word IDs
input_layer = Input(shape=(MAX_LEN,))

# Embedding layer: converts word IDs to dense vectors
# mask_zero=True tells the model to ignore padded positions
embedding = Embedding(input_dim=n_words, output_dim=EMBEDDING_DIM,
                      input_length=MAX_LEN, mask_zero=True)(input_layer)

# Bi-LSTM layer: reads sequence in both directions
# return_sequences=True means it outputs a vector for EACH token (not just the last)
bilstm = Bidirectional(LSTM(units=LSTM_UNITS, return_sequences=True))(embedding)

# Dense output layer: one softmax classifier per token position
# TimeDistributed applies the same Dense layer to each time step
output = TimeDistributed(Dense(n_tags, activation='softmax'))(bilstm)

# Compile the model
model = Model(input_layer, output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

c:\Users\avo9\AppData\Local\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 75)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 75, 50)    │    743,950 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 75)        │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 75, 128)   │     58,880 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 75, 13)    │      1,677 │ bidirectional[0]… │
│ (TimeDistributed)   │                   │            │ not_equal[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 804,507 (3.07 MB)

 Trainable params: 804,507 (3.07 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# --- Step 4: Train the model ---
print("Training Bi-LSTM model...")
history = model.fit(
    X_train_nn, y_train_nn,
    validation_split=0.1,    # use 10% of training data for validation
    batch_size=32,
    epochs=10,               # number of passes through the data
    verbose=1
)

print("\nTraining complete!")

Training Bi-LSTM model...
Epoch 1/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 25s 137ms/step - accuracy: 0.9343 - loss: 0.8529 - val_accuracy: 0.9523 - val_loss: 0.3410
Epoch 2/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 7s 96ms/step - accuracy: 0.9483 - loss: 0.3506 - val_accuracy: 0.9523 - val_loss: 0.3149
Epoch 3/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 12s 161ms/step - accuracy: 0.9482 - loss: 0.3020 - val_accuracy: 0.9523 - val_loss: 0.2883
Epoch 4/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 9s 121ms/step - accuracy: 0.9483 - loss: 0.2450 - val_accuracy: 0.9523 - val_loss: 0.2630
Epoch 5/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 9s 118ms/step - accuracy: 0.9489 - loss: 0.1955 - val_accuracy: 0.9527 - val_loss: 0.2498
Epoch 6/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 20s 258ms/step - accuracy: 0.9517 - loss: 0.1628 - val_accuracy: 0.9521 - val_loss: 0.2511
Epoch 7/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 15s 184ms/step - accuracy: 0.9560 - loss: 0.1444 - val_accuracy: 0.9525 - val_loss: 0.2634
Epoch 8/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 12s 152ms/step - accuracy: 0.9608 - l

In [13]:
# --- Step 5: Evaluate Bi-LSTM with seqeval ---
# Convert model predictions back to tag names for seqeval

y_pred_probs = model.predict(X_test_nn)
y_pred_ids = np.argmax(y_pred_probs, axis=-1)  # probabilities → class IDs
y_true_ids = np.argmax(y_test_nn, axis=-1)

# Convert IDs back to tag names, ignoring padded positions
y_true_tags = []
y_pred_tags = []

for true_seq, pred_seq, x_seq in zip(y_true_ids, y_pred_ids, X_test_nn):
    true_sent = []
    pred_sent = []
    for t, p, x in zip(true_seq, pred_seq, x_seq):
        if x != 0:  # skip padding tokens (word ID = 0)
            true_sent.append(idx2tag[t])
            pred_sent.append(idx2tag[p])
    y_true_tags.append(true_sent)
    y_pred_tags.append(pred_sent)

print("Bi-LSTM Entity-Level Evaluation:")
print("=" * 60)
print(seq_report(y_true_tags, y_pred_tags))

22/22 ━━━━━━━━━━━━━━━━━━━━ 13s 349ms/step
Bi-LSTM Entity-Level Evaluation:
               precision    recall  f1-score   support

  corporation       0.00      0.00      0.00        39
creative-work       0.00      0.00      0.00        30
        group       0.00      0.00      0.00        62
     location       0.24      0.14      0.17       117
       person       0.23      0.23      0.23       136
      product       0.00      0.00      0.00        20

    micro avg       0.20      0.12      0.15       404
    macro avg       0.08      0.06      0.07       404
 weighted avg       0.14      0.12      0.13       404



c:\Users\avo9\AppData\Local\anaconda3\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


---
## Part 5: Model Comparison

Let's compare all three approaches side-by-side.

| Approach | Features | Data Needed | Strengths | Weaknesses |
|----------|----------|-------------|-----------|------------|
| **CRF** | Hand-crafted | Small (100+) | Fast, interpretable | Can't handle OOV |
| **Bi-LSTM** | Learned embeddings | Medium (500+) | No feature engineering | Needs more data |

### Which Should You Use?
- **CRF**: When you have small data and need fast iteration
- **Bi-LSTM**: Rarely used alone today (Transformers are better)

### Next Steps
- **Notebook 3**: Use LLMs for zero-shot NER (no training data needed!)
  and build an end-to-end pipeline from annotation to deployment